In [1]:
import torch
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_parallel_coordinate, plot_contour
import yaml
import plotly
from run_with_lightning import train_and_test, train_and_test_from_yaml, load_and_test

In [2]:
torch.set_float32_matmul_precision('high') # will use tensor cores if available, otherwise will use float32 precision
seed = 42 # use 42 for reproducibility
data_config = {
    'batch_size' : 128, # use 128 for reproducibility
    'augment' : False} # whether to augment the data during training
# we will use augmentations later for CNNs

Here we compare the optimized architectures.
First, the basic fixed MLP, `MLPBasic`:

In [ ]:
architecture_type = 'mlp_basic'
# see architectures.py for the definition of MLPBasic, or below for an outline
net_params = {} # MLPBasic is a fixed architecture, so no parameters are needed
lr = 0.01 #learning rate
weight_decay = 0.001
optimizer_params = {'lr': lr, 'weight_decay': weight_decay}
lit_module_config = {'architecture_type': architecture_type, 'net_params': net_params, 'optimizer_params': optimizer_params}
train_and_test(lit_module_config=lit_module_config, data_config=data_config, seed=seed)

We got test accuracy of 0.870. This sets the baseline for us. Let's see if we can improve on the MLP architecture and optimizer with Optuna.

running `python -m optumize --mlp` will start a big Optuna optimization study, where 100 different hyperparameter configurations will be explored efficiently using TPE (tree Parzen estimators) sampler and the median pruner.

We already run the study, the results were automatically saved in `optuna_databases/mlp.db` database. The best hyperparameter configuration is automatically saved in `mlp_best_params.yaml` we can test this directly with `train_test_from_yaml('config_path'='mlp_best_params.yaml', 'data_config'=data_config)` (remember, for hyperparameter optimization we only use the train and validation dataset, but what we are really interested in is the performance on the test dataset).

But first, let's visualize the Optuna study:

In [8]:
studies_dir = "optuna_databases"
mlp_study_name = "mlp"
storage_name = f"sqlite:///{studies_dir}/{mlp_study_name}.db"
study_mlp = optuna.load_study(study_name=mlp_study_name, storage=storage_name)

In [9]:
plot_optimization_history(study_mlp)

Looks like Optuna got lucky with a good guess on the very first trial and only marginally improved from there, exploring many unsuccessful configurations inbetween.

Slice plot is a great tool to see which parameters work best and which don't:

In [10]:
plot_slice(study_mlp)

We clearly found the best learning rate. Weight decay and number of layers don't seem to matter too much, but perhaps we were too conservative with maximum number of hidden units per hidden layer. We see clear improvement the more hidden units we use up until the maximum we set at 256 in the search space.

This is typical when doing a first hyperparameter optimization study on a dataset and architecture you are not familiar with. We ran another study where we fixed the learning rate at 1e-3, weight decay at 1e-5, restricted number of hidden layers to 4 maximum, but relaxed the maximum number of hidden units up to 1024. Let's see how that went:

In [11]:
mlp2_study_name = "mlp2"
storage_name = f"sqlite:///{studies_dir}/{mlp2_study_name}.db"
study_mlp2 = optuna.load_study(study_name=mlp2_study_name, storage=storage_name)

In [12]:
plot_slice(study_mlp2)


While we still see improvement with increasing the number of hidden units, the improvement is very small, still just above 0.90, at a high computational cost. The differences are well within random variation from run to run.
This means we reached the limit of what MLP architecture can give us on this dataset. The best configuration was saved automatically in `mlp3_best_params.yaml`. Let's see how the best configuration does on the test dataset:

In [ ]:
train_and_test_from_yaml(config_path='mlp3_best_params.yaml', data_config=data_config, seed=seed)

With hyperparameter optimization, we went from under 0.870 to 0.894 test accuracy for the MLP architecture.

Let's see what a basic fixed CNN can do with a good guess for learning rate and weight decay. CNNs are more powerful, so we will add data augmentation here: random flips and small scaling during training. See the datamodule definition in `lightning_definitions.py` for full details.

In [ ]:
architecture_type = 'cnn_basic'
# see architectures.py for the definition of CNNBasic, or below for an outline
net_params = {} # CNNBasic is a fixed architecture, so no parameters are needed
lr = 0.01 # learning rate
weight_decay = 0.001
optimizer_params = {'lr': lr, 'weight_decay': weight_decay}
lit_module_config = {'architecture_type': architecture_type, 'net_params': net_params, 'optimizer_params': optimizer_params}
data_config = {
    'batch_size' : 128, # use 128 for reproducibility
    'augment' : True} # turn on augmentation for CNNs
train_and_test(lit_module_config=lit_module_config, data_config=data_config, seed=seed)

This is not fully reproducible on CUDA, but should give about 0.88 test accuracy. So, better than the basic MLP architecture we started with, but worse than the optimized MLP.

Hopefully we can improve on this with hyperparameter optimization.

running `python -m optumize --cnn` will again start a big Optuna optimization study for the CNN architecture.

We already run the study, the results were automatically saved in `optuna_databases/cnn2.db` database. The best hyperparameter configuration is automatically saved in `cnn2_best_params.yaml`. Let's test the best configuration.

In [ ]:
train_and_test_from_yaml(config_path='cnn2_best_params.yaml', data_config=data_config, seed=seed)

We got over 0.94 test accuracy, but the model is huge! 17.3M parameters, 69MB estimated model size. Let's visualize the Optuna study and see if we can get something cheaper.

In [ ]:
train_and_test_from_yaml(config_path='cnn2_smaller.yaml', data_config=data_config, seed=seed)

cnn2_smaller is 4 times smaller but gives nearly the same test accuracy at 0.94

In [5]:
train_and_test_from_yaml(config_path='cnn2_small.yaml', data_config=data_config, seed=seed)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type               | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | net      | CNN2               | 805 K  | train | 0    
1 | loss     | CrossEntropyLoss   | 0      | train | 0    
2 | accuracy | MulticlassAccuracy | 0      | train | 0    
----------------------------------------------------------------
805 K     Trainable params
0         Non-trainable params
805 K     Total params
3.224     Total estimated model params size (MB)
27        Modules in train mode
0         Modules in eval mode
0         Total Flops


LitModule(
  (net): CNN2(
    (conv_net): Sequential(
      (0): Sequential(
        (0): Conv2d(1, 90, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (1): BatchNorm2d(90, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(90, 90, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (4): BatchNorm2d(90, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (5): ReLU()
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
      (1): Sequential(
        (0): Conv2d(90, 180, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (1): BatchNorm2d(180, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(180, 180, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (4): BatchNorm2d(180, eps=1e-05, momentum=0.1, affine=True, bias=

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Restoring states from the checkpoint path at /home/denis/Coding/nas-fashion-mnist/lightning_logs/version_1183/checkpoints/epoch=12-step=4875.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/denis/Coding/nas-fashion-mnist/lightning_logs/version_1183/checkpoints/epoch=12-step=4875.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.9243797063827515
        test_loss           0.24864192306995392
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.24864192306995392, 'test_accuracy': 0.9243797063827515}]

The smallest reasonable architecture from the study, saved in `cnn2_small.yaml` is only 805k parameters, but performance dropped to below 0.93.

Even smaller configuration at 298k prameters from another study is saved in `cnn_batch_128_no_prune_best_params.yaml`. Test accuracy is just 0.92.

In [ ]:
cnn2_study_name = "cnn2"
storage_name = f"sqlite:///{studies_dir}/{cnn2_study_name}.db"
study_cnn2 = optuna.load_study(study_name=cnn2_study_name, storage=storage_name)

In [ ]:
train_and_test_from_yaml(config_path='cnn_batch_128_no_prune_best_params.yaml', data_config=data_config, seed=seed)

In [7]:
train_and_test_from_yaml(config_path='cnn_64_best_params.yaml', data_config=data_config, seed=seed)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type               | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | net      | CNN2               | 408 K  | train | 0    
1 | loss     | CrossEntropyLoss   | 0      | train | 0    
2 | accuracy | MulticlassAccuracy | 0      | train | 0    
----------------------------------------------------------------
408 K     Trainable params
0         Non-trainable params
408 K     Total params
1.634     Total estimated model params size (MB)
27        Modules in train mode
0         Modules in eval mode
0         Total Flops


LitModule(
  (net): CNN2(
    (conv_net): Sequential(
      (0): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (5): ReLU()
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
      (1): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
        (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Restoring states from the checkpoint path at /home/denis/Coding/nas-fashion-mnist/lightning_logs/version_1185/checkpoints/epoch=13-step=5250.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/denis/Coding/nas-fashion-mnist/lightning_logs/version_1185/checkpoints/epoch=13-step=5250.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.9262968897819519
        test_loss           0.25474539399147034
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.25474539399147034, 'test_accuracy': 0.9262968897819519}]

In [ ]:
plot_slice(study_cnn2)